In [1]:
import sys
sys.path.append('../datasets')
sys.path.append('../algorithms')
sys.path.append('../test')


import torch

import matplotlib.pyplot as plt


from pansharpening import PANDataset
from alg3 import PANTVGradAlignement
from torchvision import transforms
from math import sqrt
from nabla import nabla

/home/ndiayem/Documents/spectral-spatial/env/lib/python3.11/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
#Define device (default is "cpu")
device = "cpu" 

    # Define dtype
dtype = torch.float64
crop_size = 256

    # Define random seed
seed = 42
torch.manual_seed(seed)

    # Define data path
data_path = '/home/ndiayem/Documents/spectral-spatial/data/harvard.zarr'

In [3]:
val_transform = transforms.Compose([transforms.CenterCrop(crop_size)]) # Transforms a the input data to torch tensors
dataset = PANDataset(root_dir=data_path, split='train' ,transform=val_transform,normalize= True,scale=2,sigma=0.1,device='cpu', size=crop_size)

In [4]:
idx = 17
X = dataset[idx]
X = X.unsqueeze(0)

In [5]:
X.shape

torch.Size([1, 31, 256, 256])

In [6]:
# Matrice de transformation
Y_H = dataset.simulate_low_res_hsi(X)  
Y_M = dataset.get_panchromatic(X)

In [ ]:
grad_panc = nabla(Y_M)
params = {
    'max_iter': 100,         # niters → max_iter (nom attendu par TVPrior)
    'lmbda': 1e-2,          # paramètre supplémentaire
    'theta': 1 ,            # paramètre de régularisation
    'sigma': 2,                  # sigma = gain (gain=2)   
    'tau': 0.99/2,            # tau = 0.99 / gain (calculé)
    'grad_panc' : grad_panc, # le gradient de la panchromatique                       
    # 'weight_fun' : lambda normp, alpha: torch.stack((torch.ones_like(normp), ... , dim=-1).transpose(4,5).squeeze(-1)..
}


In [10]:
A,A_adj, R, R_adj = dataset.get_operators()

solver_gp = PANTVGradAlignement(
    A=A,
    Aadj=A_adj,
    spectral_op = dataset.spectral_op,
    spectral_op_t = dataset.spectral_op_t,
    max_iter=100,
    lmbda=1e-2,
    lmbda_m=5,
    tol=1e-7,
    scale=dataset.scale,
    p = 1,
    q = 1,
    r = 1,
    verbose=True,
    params = params
)

In [11]:
U_cb, costs_gp = solver_gp(Y_H, Y_M)


Début de l'optimisation:
It    | Coût total   | Data H       | Data M       | TV           | ΔU          
--------------------------------------------------------------------------------
0     | 4.616e+03    | 3.120e+03    | 1.467e+03    | 2.986e+01    | 1.114e-02   
10    | 4.296e+03    | 2.907e+03    | 1.356e+03    | 3.266e+01    | 9.663e-03   
20    | 4.000e+03    | 2.710e+03    | 1.254e+03    | 3.539e+01    | 8.495e-03   
30    | 3.726e+03    | 2.528e+03    | 1.160e+03    | 3.792e+01    | 7.546e-03   
40    | 3.472e+03    | 2.358e+03    | 1.074e+03    | 4.028e+01    | 6.760e-03   
50    | 3.238e+03    | 2.201e+03    | 9.945e+02    | 4.250e+01    | 6.099e-03   
60    | 3.021e+03    | 2.055e+03    | 9.214e+02    | 4.459e+01    | 5.537e-03   
70    | 2.820e+03    | 1.919e+03    | 8.540e+02    | 4.655e+01    | 5.052e-03   
80    | 2.634e+03    | 1.794e+03    | 7.920e+02    | 4.839e+01    | 4.632e-03   
90    | 2.462e+03    | 1.677e+03    | 7.349e+02    | 5.012e+01    | 4.263e-03   
99

In [ ]:
U_cb.shape

In [ ]:
rgb = dataset.rgb_index
x_rgb = X[0, rgb,...].cpu().numpy().transpose(1, 2, 0)
x_rgb = (x_rgb - x_rgb.min())/(x_rgb.max() - x_rgb.min())

y_rgb = Y_H[0, rgb, ...].cpu().numpy().transpose(1, 2, 0)
y_rgb = (y_rgb - y_rgb.min())/(y_rgb.max() - y_rgb.min())

z_rgb = U_cb[0, rgb, ...].cpu().numpy().transpose(1, 2, 0)
z_rgb = (z_rgb - z_rgb.min())/(z_rgb.max() - z_rgb.min())

# plt.figure(figsize=(15, 5))
# plt.subplot(131)
plt.figure(figsize=(10,10))
plt.imshow(x_rgb)
# plt.title('RGB')
plt.axis('off')

# plt.subplot(132)
plt.figure(figsize=(10,10))
plt.imshow(Y_M[0, 0, ...].cpu().numpy(), cmap='gray')
# plt.title('Panchromatic')
plt.axis('off')

# plt.subplot(133)
plt.figure(figsize=(10,10))
plt.imshow(y_rgb)
# plt.title('Noisy RGB')
plt.axis('off')

# plt.subplot(133)
plt.figure(figsize=(10,10))
plt.imshow(z_rgb)
# plt.title('Noisy RGB')
plt.axis('off')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données d'exemple (à adapter avec vos données réelles)
cout_total = costs_gp

# Création du vecteur d'itérations de même longueur que cout_total
iterations = np.arange(0, len(cout_total))  # Pas de 10 entre les mesures

# Vérification des dimensions
print(f"Dimensions vérifiées: iterations {iterations.shape}, cout_total {len(cout_total)}")

# Création du graphique
plt.figure(figsize=(12, 6))
plt.semilogy(iterations, cout_total, 'b-', linewidth=1.5, label='Coût total')

# Personnalisation
plt.xlabel('Itérations', fontsize=12)
plt.ylabel('Coût total (échelle log10)', fontsize=12)
plt.title('Évolution du Coût Total', fontsize=14)
plt.grid(True, which="both", linestyle='--', alpha=0.6)

# Affichage
plt.legend()
plt.tight_layout()
plt.show()